In [1]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "muhlenbeck2016differences")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "data_not_original.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)



df['study_id']="muhlenbeck2016differences"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
df = df[~df['study'].isin(['namibians', 'germans'])]


df.rename(columns={"proband": "participant",
                   "age":"age_in_years"}, inplace=True)
# df.columns
# df['participant'].unique()

In [3]:
participant_list = [['ap_vp1','batak'],
        ['ap_vp2', 'bimbo'],
        ['ap_vp3', 'dokana'],
        ['ap_vp4', 'padana'],
        ['ap_vp5', 'pini'],
        ['ap_vp6', 'raja'],    
        ['ap_vp7', 'suaq' ],    
        ['ap_vp8', 'tanah']]
for x,y in participant_list:
    df['participant'].replace(x, y, inplace=True, regex=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
df= df.merge(apedf,left_on='participant', right_on='name', how='left')

In [4]:


studyID_standardized=df[[ 'study_id', 'participant','age_in_years', 'sex','species', 
                     #     'study', 
       'timestamp', 'fixationduration', 'aoiids', 'aoinames', 'fixmark',
       'fixmarknew', 'stimuliname', 'picnumb', 'fixationpointx',
       'fixationpointy' ]]
comp_out_path_stand = os.path.join(out_pathway, 'muhlenbeck2016differences_standardized.csv')
studyID_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig',sep=',' ,index=False)


names =studyID_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
studyID_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'muhlenbeck2016differences_glossary.csv')
studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig',sep=',', index=False)
